In [0]:
%sql
create materialized view primeinsurance.gold_layer.GOLD_deduplicated_customers as 
SELECT customer_id, COUNT(DISTINCT region) AS region_count
FROM primeinsurance.gold_layer.dim_customer
GROUP BY customer_id
HAVING COUNT(DISTINCT region) > 1;

result
The operation was successfully executed.


In [0]:
%sql
create materialized view primeinsurance.gold_layer.GOLD_claim_rejection_rate as
SELECT
 dc.region,
 fp.policy_csl,
 SUM(fc.claim_rejected) / COUNT(*) AS rejection_rate
FROM primeinsurance.gold_layer.dim_policy fp
Join primeinsurance.gold_layer.fact_claims fc on fc.claim_id = fp.policy_number
Join primeinsurance.gold_layer.dim_customer dc on dc.customer_id = fp.customer_id
GROUP BY dc.region, fp.policy_csl;



result
The operation was successfully executed.


In [0]:
%sql
create materialized view primeinsurance.gold_layer.GOLD_vw_unsold_cars_by_model_reg as 
SELECT
 dc.model,
 fs.region,
 COUNT(*) AS unsold_count
FROM primeinsurance.gold_layer.fact_sales fs
JOIN primeinsurance.gold_layer.dim_car dc ON fs.car_id = dc.car_id
WHERE fs.sold_on IS NULL
GROUP BY dc.model, fs.region;


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-1523715457219126>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'create materialized view primeinsurance.gold_layer.GOLD_vw_unsold_cars_by_model_reg as \nSELECT\n dc.model,\n fs.region,\n COUNT(*) AS unsold_count\nFROM primeinsurance.gold_layer.fact_sales fs\nJOIN primeinsurance.gold_layer.dim_car dc ON fs.car_id = dc.car_id\nWHERE fs.sold_on IS NULL\nGROUP BY dc.model, fs.region;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python to

In [0]:
%sql
create materialized view primeinsurance.gold_layer.GOLD_vw_aging_bucket as 
SELECT
 region,
 CASE 
   WHEN (ad_placed_on - sold_on)  <= 30 THEN '0-30'
   WHEN (ad_placed_on - sold_on)   <= 60 THEN '30-60'
   ELSE '60+'
 END AS aging_bucket,
 COUNT(*)
FROM primeinsurance.gold_layer.fact_sales
WHERE sold_on IS NULL
GROUP BY region, aging_bucket;


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7500902637420866>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "create materialized view primeinsurance.gold_layer.GOLD_vw_aging_bucket as \nSELECT\n region,\n CASE \n   WHEN (ad_placed_on - sold_on)  <= 30 THEN '0-30'\n   WHEN (ad_placed_on - sold_on)   <= 60 THEN '30-60'\n   ELSE '60+'\n END AS aging_bucket,\n COUNT(*)\nFROM primeinsurance.gold_layer.fact_sales\nWHERE sold_on IS NULL\nGROUP BY region, aging_bucket;\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_sile